<a href="https://colab.research.google.com/github/Munees-2811/face-detection-desk/blob/main/facedetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# ================================================================
# YOLO11 ATTENTION AND SLEEP MONITORING - GOOGLE COLAB
#
# Detects:
#   1. WATCHING SYSTEM
#   2. WATCHING OTHER SIDE - LEFT
#   3. WATCHING OTHER SIDE - RIGHT
#   4. LOOKING DOWN
#   5. SLEEPING
#   6. NO FACE
#
# Allow camera permission when the browser asks.
# Click "Stop Camera" to finish.
# ================================================================


# ------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------

!pip install -q -U ultralytics huggingface_hub mediapipe opencv-python


# ------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------

import os
import cv2
import time
import math
import base64
import urllib.request
import numpy as np
import mediapipe as mp

from ultralytics import YOLO
from huggingface_hub import hf_hub_download

from google.colab.output import eval_js
from IPython.display import display, Javascript


# ------------------------------------------------
# 3. DOWNLOAD YOLO11 FACE MODEL
# ------------------------------------------------

print("Downloading YOLO11 face-detection model...")

YOLO_MODEL_PATH = hf_hub_download(
    repo_id="AdamCodd/YOLOv11n-face-detection",
    filename="model.pt"
)

print("YOLO model downloaded:")
print(YOLO_MODEL_PATH)


# ------------------------------------------------
# 4. DOWNLOAD MEDIAPIPE FACE LANDMARKER MODEL
# ------------------------------------------------

LANDMARK_MODEL_PATH = "/content/face_landmarker.task"

LANDMARK_MODEL_URL = (
    "https://storage.googleapis.com/"
    "mediapipe-models/face_landmarker/"
    "face_landmarker/float16/1/"
    "face_landmarker.task"
)

if not os.path.exists(LANDMARK_MODEL_PATH):

    print("Downloading MediaPipe face-landmark model...")

    urllib.request.urlretrieve(
        LANDMARK_MODEL_URL,
        LANDMARK_MODEL_PATH
    )

if not os.path.exists(LANDMARK_MODEL_PATH):
    raise FileNotFoundError(
        "MediaPipe model download failed."
    )

print("MediaPipe model downloaded:")
print(LANDMARK_MODEL_PATH)


# ------------------------------------------------
# 5. LOAD YOLO11 FACE MODEL
# ------------------------------------------------

face_detector = YOLO(YOLO_MODEL_PATH)

print("YOLO11 face detector loaded successfully.")


# ------------------------------------------------
# 6. CREATE MEDIAPIPE FACE LANDMARKER
# ------------------------------------------------

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

landmarker_options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=LANDMARK_MODEL_PATH
    ),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True
)

face_landmarker = FaceLandmarker.create_from_options(
    landmarker_options
)

print("MediaPipe Face Landmarker loaded successfully.")


# ------------------------------------------------
# 7. SETTINGS
# ------------------------------------------------

YOLO_CONFIDENCE = 0.40
YOLO_IMAGE_SIZE = 320

# Lower EAR means the eyes are more closed.
# Change between 0.18 and 0.24 if needed.
EYE_CLOSED_THRESHOLD = 0.20

# Eyes must remain closed this long to label sleeping.
SLEEP_SECONDS = 1.5

# Head-turn thresholds in degrees.
YAW_THRESHOLD = 18

# Looking-down threshold.
PITCH_DOWN_THRESHOLD = 18

# Amount of padding around YOLO face crop.
FACE_PADDING = 25


# ------------------------------------------------
# 8. MEDIAPIPE LANDMARK INDEXES
# ------------------------------------------------

# Six points around each eye.
LEFT_EYE_POINTS = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_POINTS = [362, 385, 387, 263, 373, 380]

# Head-pose points.
NOSE_TIP = 1
CHIN = 152
LEFT_EYE_CORNER = 33
RIGHT_EYE_CORNER = 263
LEFT_MOUTH_CORNER = 61
RIGHT_MOUTH_CORNER = 291


# ------------------------------------------------
# 9. UTILITY FUNCTIONS
# ------------------------------------------------

def euclidean_distance(point1, point2):
    """Calculate distance between two image points."""

    return math.sqrt(
        (point1[0] - point2[0]) ** 2 +
        (point1[1] - point2[1]) ** 2
    )


def get_pixel_point(landmark, width, height):
    """Convert a normalized MediaPipe landmark to pixels."""

    x = int(landmark.x * width)
    y = int(landmark.y * height)

    return np.array([x, y], dtype=np.float64)


def calculate_eye_aspect_ratio(
    landmarks,
    eye_indexes,
    width,
    height
):
    """
    Calculate Eye Aspect Ratio.

    The vertical opening is compared with horizontal eye width.
    """

    points = [
        get_pixel_point(
            landmarks[index],
            width,
            height
        )
        for index in eye_indexes
    ]

    # Vertical distances.
    vertical_1 = euclidean_distance(
        points[1],
        points[5]
    )

    vertical_2 = euclidean_distance(
        points[2],
        points[4]
    )

    # Horizontal distance.
    horizontal = euclidean_distance(
        points[0],
        points[3]
    )

    if horizontal == 0:
        return 0.0

    ear = (
        vertical_1 + vertical_2
    ) / (2.0 * horizontal)

    return ear


def estimate_head_pose(
    landmarks,
    image_width,
    image_height
):
    """
    Estimate head pitch, yaw and roll using solvePnP.
    """

    image_points = np.array(
        [
            get_pixel_point(
                landmarks[NOSE_TIP],
                image_width,
                image_height
            ),
            get_pixel_point(
                landmarks[CHIN],
                image_width,
                image_height
            ),
            get_pixel_point(
                landmarks[LEFT_EYE_CORNER],
                image_width,
                image_height
            ),
            get_pixel_point(
                landmarks[RIGHT_EYE_CORNER],
                image_width,
                image_height
            ),
            get_pixel_point(
                landmarks[LEFT_MOUTH_CORNER],
                image_width,
                image_height
            ),
            get_pixel_point(
                landmarks[RIGHT_MOUTH_CORNER],
                image_width,
                image_height
            )
        ],
        dtype=np.float64
    )

    # Approximate generic 3D face coordinates.
    model_points = np.array(
        [
            (0.0, 0.0, 0.0),          # Nose
            (0.0, -330.0, -65.0),     # Chin
            (-225.0, 170.0, -135.0),  # Left eye
            (225.0, 170.0, -135.0),   # Right eye
            (-150.0, -150.0, -125.0), # Left mouth
            (150.0, -150.0, -125.0)   # Right mouth
        ],
        dtype=np.float64
    )

    focal_length = image_width

    center = (
        image_width / 2,
        image_height / 2
    )

    camera_matrix = np.array(
        [
            [focal_length, 0, center[0]],
            [0, focal_length, center[1]],
            [0, 0, 1]
        ],
        dtype=np.float64
    )

    distortion_coefficients = np.zeros(
        (4, 1),
        dtype=np.float64
    )

    success, rotation_vector, translation_vector = (
        cv2.solvePnP(
            model_points,
            image_points,
            camera_matrix,
            distortion_coefficients,
            flags=cv2.SOLVEPNP_ITERATIVE
        )
    )

    if not success:
        return 0.0, 0.0, 0.0

    rotation_matrix, _ = cv2.Rodrigues(
        rotation_vector
    )

    angles = cv2.RQDecomp3x3(
        rotation_matrix
    )[0]

    pitch = float(angles[0])
    yaw = float(angles[1])
    roll = float(angles[2])

    return pitch, yaw, roll


def draw_eye_points(
    image,
    landmarks,
    indexes,
    width,
    height
):
    """Draw eye landmark points."""

    for index in indexes:

        point = get_pixel_point(
            landmarks[index],
            width,
            height
        )

        cv2.circle(
            image,
            (int(point[0]), int(point[1])),
            2,
            (255, 255, 0),
            -1
        )


def add_status_panel(
    frame,
    status,
    ear,
    pitch,
    yaw,
    face_count
):
    """Draw activity information on the frame."""

    overlay = frame.copy()

    cv2.rectangle(
        overlay,
        (0, 0),
        (frame.shape[1], 145),
        (0, 0, 0),
        -1
    )

    frame = cv2.addWeighted(
        overlay,
        0.65,
        frame,
        0.35,
        0
    )

    if status == "SLEEPING":
        status_color = (0, 0, 255)

    elif "OTHER SIDE" in status:
        status_color = (0, 165, 255)

    elif status == "LOOKING DOWN":
        status_color = (0, 165, 255)

    elif status == "WATCHING SYSTEM":
        status_color = (0, 255, 0)

    else:
        status_color = (0, 0, 255)

    cv2.putText(
        frame,
        status,
        (15, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.85,
        status_color,
        2
    )

    cv2.putText(
        frame,
        f"Eye ratio: {ear:.3f}",
        (15, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1
    )

    cv2.putText(
        frame,
        f"Head pitch: {pitch:.1f}",
        (15, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1
    )

    cv2.putText(
        frame,
        f"Head yaw: {yaw:.1f}",
        (15, 115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1
    )

    cv2.putText(
        frame,
        f"Faces: {face_count}",
        (15, 140),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1
    )

    return frame


# ------------------------------------------------
# 10. BROWSER WEBCAM JAVASCRIPT
# ------------------------------------------------

def start_camera():

    display(Javascript("""
    async function startAttentionCamera() {

        const oldContainer =
            document.getElementById(
                "attention-camera-container"
            );

        if (oldContainer) {
            oldContainer.remove();
        }

        const container =
            document.createElement("div");

        container.id =
            "attention-camera-container";

        const title =
            document.createElement("h3");

        title.innerText =
            "YOLO11 Attention Monitoring";

        const video =
            document.createElement("video");

        video.id = "attention-video";
        video.width = 480;
        video.height = 360;
        video.autoplay = true;
        video.playsInline = true;
        video.muted = true;
        video.style.border =
            "2px solid black";

        const output =
            document.createElement("img");

        output.id = "attention-output";
        output.width = 480;
        output.height = 360;
        output.style.border =
            "3px solid green";

        const canvas =
            document.createElement("canvas");

        canvas.id = "attention-canvas";
        canvas.width = 480;
        canvas.height = 360;
        canvas.style.display = "none";

        const stopButton =
            document.createElement("button");

        stopButton.innerText =
            "Stop Camera";

        stopButton.style.padding =
            "10px 20px";

        stopButton.style.margin =
            "10px";

        stopButton.style.fontSize =
            "16px";

        stopButton.onclick = function() {

            window.attentionCameraRunning =
                false;

            if (window.attentionStream) {

                window.attentionStream
                    .getTracks()
                    .forEach(
                        track => track.stop()
                    );
            }

            stopButton.innerText =
                "Camera Stopped";
        };

        const originalHeading =
            document.createElement("p");

        originalHeading.innerText =
            "Original webcam";

        const resultHeading =
            document.createElement("p");

        resultHeading.innerText =
            "Detection result";

        container.appendChild(title);
        container.appendChild(
            originalHeading
        );
        container.appendChild(video);
        container.appendChild(
            resultHeading
        );
        container.appendChild(output);
        container.appendChild(
            document.createElement("br")
        );
        container.appendChild(stopButton);
        container.appendChild(canvas);

        document.body.appendChild(
            container
        );

        try {

            const stream =
                await navigator.mediaDevices
                    .getUserMedia({
                        video: {
                            width: 480,
                            height: 360,
                            facingMode: "user"
                        },
                        audio: false
                    });

            video.srcObject = stream;

            await video.play();

            window.attentionStream =
                stream;

            window.attentionCameraRunning =
                true;

        } catch (error) {

            window.attentionCameraRunning =
                false;

            alert(
                "Camera could not start: " +
                error.message
            );
        }
    }


    function captureAttentionFrame() {

        if (
            !window.attentionCameraRunning
        ) {
            return null;
        }

        const video =
            document.getElementById(
                "attention-video"
            );

        const canvas =
            document.getElementById(
                "attention-canvas"
            );

        if (
            !video ||
            !canvas ||
            video.readyState < 2
        ) {
            return null;
        }

        const context =
            canvas.getContext("2d");

        // Mirror the camera image naturally.
        context.save();
        context.translate(
            canvas.width,
            0
        );
        context.scale(-1, 1);

        context.drawImage(
            video,
            0,
            0,
            canvas.width,
            canvas.height
        );

        context.restore();

        return canvas.toDataURL(
            "image/jpeg",
            0.70
        );
    }


    function showAttentionResult(
        imageData
    ) {

        const output =
            document.getElementById(
                "attention-output"
            );

        if (output) {
            output.src = imageData;
        }
    }


    function isAttentionCameraRunning() {

        return (
            window.attentionCameraRunning
            === true
        );
    }


    function stopAttentionCamera() {

        window.attentionCameraRunning =
            false;

        if (window.attentionStream) {

            window.attentionStream
                .getTracks()
                .forEach(
                    track => track.stop()
                );
        }
    }


    startAttentionCamera();
    """))


# ------------------------------------------------
# 11. IMAGE-CONVERSION FUNCTIONS
# ------------------------------------------------

def browser_image_to_cv2(data_url):

    if data_url is None:
        return None

    encoded_data = data_url.split(",")[1]

    decoded_data = base64.b64decode(
        encoded_data
    )

    image_array = np.frombuffer(
        decoded_data,
        dtype=np.uint8
    )

    return cv2.imdecode(
        image_array,
        cv2.IMREAD_COLOR
    )


def cv2_image_to_browser(image):

    success, encoded_image = cv2.imencode(
        ".jpg",
        image,
        [cv2.IMWRITE_JPEG_QUALITY, 70]
    )

    if not success:
        return None

    base64_image = base64.b64encode(
        encoded_image
    ).decode("utf-8")

    return (
        "data:image/jpeg;base64," +
        base64_image
    )


# ------------------------------------------------
# 12. START REAL-TIME MONITORING
# ------------------------------------------------

start_camera()

print()
print("Allow camera permission in the browser.")
print("Click 'Stop Camera' to finish.")
print()
print("Statuses:")
print("  WATCHING SYSTEM")
print("  WATCHING OTHER SIDE - LEFT")
print("  WATCHING OTHER SIDE - RIGHT")
print("  LOOKING DOWN")
print("  SLEEPING")
print("  NO FACE")
print()

time.sleep(3)

eyes_closed_start_time = None

# Smooth head-pose values to reduce label flickering.
smoothed_yaw = 0.0
smoothed_pitch = 0.0

SMOOTHING = 0.75


try:

    while True:

        camera_running = eval_js(
            "isAttentionCameraRunning()"
        )

        if not camera_running:
            print("Camera stopped.")
            break

        frame_data = eval_js(
            "captureAttentionFrame()"
        )

        if frame_data is None:
            time.sleep(0.05)
            continue

        frame = browser_image_to_cv2(
            frame_data
        )

        if frame is None:
            continue

        frame_height, frame_width = (
            frame.shape[:2]
        )

        # ----------------------------------------
        # YOLO11 FACE DETECTION
        # ----------------------------------------

        yolo_results = face_detector.predict(
            source=frame,
            conf=YOLO_CONFIDENCE,
            imgsz=YOLO_IMAGE_SIZE,
            verbose=False
        )

        boxes = yolo_results[0].boxes

        face_count = (
            len(boxes)
            if boxes is not None
            else 0
        )

        status = "NO FACE"
        average_ear = 0.0
        pitch = 0.0
        yaw = 0.0

        if face_count > 0:

            # Use the largest detected face.
            largest_box = None
            largest_area = 0

            for box in boxes:

                x1, y1, x2, y2 = map(
                    int,
                    box.xyxy[0].tolist()
                )

                area = (
                    max(0, x2 - x1) *
                    max(0, y2 - y1)
                )

                if area > largest_area:

                    largest_area = area
                    largest_box = (
                        x1,
                        y1,
                        x2,
                        y2
                    )

            if largest_box is not None:

                x1, y1, x2, y2 = (
                    largest_box
                )

                # Add padding around face.
                crop_x1 = max(
                    0,
                    x1 - FACE_PADDING
                )

                crop_y1 = max(
                    0,
                    y1 - FACE_PADDING
                )

                crop_x2 = min(
                    frame_width,
                    x2 + FACE_PADDING
                )

                crop_y2 = min(
                    frame_height,
                    y2 + FACE_PADDING
                )

                face_crop = frame[
                    crop_y1:crop_y2,
                    crop_x1:crop_x2
                ]

                if face_crop.size > 0:

                    crop_height, crop_width = (
                        face_crop.shape[:2]
                    )

                    rgb_face = cv2.cvtColor(
                        face_crop,
                        cv2.COLOR_BGR2RGB
                    )

                    mediapipe_image = mp.Image(
                        image_format=(
                            mp.ImageFormat.SRGB
                        ),
                        data=rgb_face
                    )

                    landmark_result = (
                        face_landmarker.detect(
                            mediapipe_image
                        )
                    )

                    if (
                        landmark_result
                        .face_landmarks
                    ):

                        landmarks = (
                            landmark_result
                            .face_landmarks[0]
                        )

                        left_ear = (
                            calculate_eye_aspect_ratio(
                                landmarks,
                                LEFT_EYE_POINTS,
                                crop_width,
                                crop_height
                            )
                        )

                        right_ear = (
                            calculate_eye_aspect_ratio(
                                landmarks,
                                RIGHT_EYE_POINTS,
                                crop_width,
                                crop_height
                            )
                        )

                        average_ear = (
                            left_ear + right_ear
                        ) / 2.0

                        raw_pitch, raw_yaw, roll = (
                            estimate_head_pose(
                                landmarks,
                                crop_width,
                                crop_height
                            )
                        )

                        # Smooth noisy angle values.
                        smoothed_yaw = (
                            SMOOTHING *
                            smoothed_yaw
                            +
                            (1 - SMOOTHING) *
                            raw_yaw
                        )

                        smoothed_pitch = (
                            SMOOTHING *
                            smoothed_pitch
                            +
                            (1 - SMOOTHING) *
                            raw_pitch
                        )

                        yaw = smoothed_yaw
                        pitch = smoothed_pitch

                        current_time = (
                            time.time()
                        )

                        # ------------------------
                        # SLEEP DETECTION
                        # ------------------------

                        if (
                            average_ear <
                            EYE_CLOSED_THRESHOLD
                        ):

                            if (
                                eyes_closed_start_time
                                is None
                            ):
                                eyes_closed_start_time = (
                                    current_time
                                )

                            closed_duration = (
                                current_time -
                                eyes_closed_start_time
                            )

                        else:

                            eyes_closed_start_time = (
                                None
                            )

                            closed_duration = 0.0

                        # ------------------------
                        # ACTIVITY CLASSIFICATION
                        # ------------------------

                        if (
                            closed_duration
                            >= SLEEP_SECONDS
                        ):

                            status = "SLEEPING"

                        elif (
                            yaw >
                            YAW_THRESHOLD
                        ):

                            status = (
                                "WATCHING OTHER "
                                "SIDE - RIGHT"
                            )

                        elif (
                            yaw <
                            -YAW_THRESHOLD
                        ):

                            status = (
                                "WATCHING OTHER "
                                "SIDE - LEFT"
                            )

                        elif (
                            pitch >
                            PITCH_DOWN_THRESHOLD
                        ):

                            status = (
                                "LOOKING DOWN"
                            )

                        else:

                            status = (
                                "WATCHING SYSTEM"
                            )

                        # Draw eye points on crop.
                        draw_eye_points(
                            face_crop,
                            landmarks,
                            LEFT_EYE_POINTS,
                            crop_width,
                            crop_height
                        )

                        draw_eye_points(
                            face_crop,
                            landmarks,
                            RIGHT_EYE_POINTS,
                            crop_width,
                            crop_height
                        )

                    else:

                        status = (
                            "FACE LANDMARKS "
                            "NOT FOUND"
                        )

                        eyes_closed_start_time = (
                            None
                        )

                # Draw YOLO face box.
                if status == "SLEEPING":
                    box_color = (0, 0, 255)

                elif status == "WATCHING SYSTEM":
                    box_color = (0, 255, 0)

                else:
                    box_color = (0, 165, 255)

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    box_color,
                    2
                )

        else:

            eyes_closed_start_time = None

        frame = add_status_panel(
            frame,
            status,
            average_ear,
            pitch,
            yaw,
            face_count
        )

        browser_output = (
            cv2_image_to_browser(frame)
        )

        if browser_output is not None:

            eval_js(
                "showAttentionResult("
                f"'{browser_output}'"
                ")"
            )


except KeyboardInterrupt:

    eval_js(
        "stopAttentionCamera()"
    )

    print("Monitoring stopped manually.")


except Exception as error:

    try:
        eval_js(
            "stopAttentionCamera()"
        )
    except Exception:
        pass

    print("Error:", error)


finally:

    try:
        face_landmarker.close()
    except Exception:
        pass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.3 MB/s eta 0:00:00


YOLO model downloaded:
/root/.cache/huggingface/hub/models--AdamCodd--YOLOv11n-face-detection/snapshots/0e97fea5eacb1460b35725c929d813c7095841b5/model.pt
MediaPipe model downloaded:
/content/face_landmarker.task
YOLO11 face detector loaded successfully.
MediaPipe Face Landmarker loaded successfully.


<IPython.core.display.Javascript object>


Allow camera permission in the browser.
Click 'Stop Camera' to finish.

Statuses:
  WATCHING SYSTEM
  WATCHING OTHER SIDE - LEFT
  WATCHING OTHER SIDE - RIGHT
  LOOKING DOWN
  SLEEPING
  NO FACE

Monitoring stopped manually.
